In [1]:
#import library
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats

#import sklearn library
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

#import wandb library
import wandb

#import joblib library
import joblib as jb

In [2]:
# Cố định random state để kết quả reproducible
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

In [3]:
# Cấu hình WandB
WANDB_PROJECT = "heart-disease-classification"
WANDB_ENTITY  = "phat-05"

In [4]:
#hàm load data set để handle error và tái sử dụng
def load_dataset(dataset_path):
    """
    Kiểm tra và lấy dữ liệu từ file csv 
    Args:
        dataset_path (str): đường dẫn file csv 
    Returns:
        pandas.dataframe: bảng dữ liệu đã được đọc
    """
    if not os.path.exists(dataset_path):
        print("Không tìm thấy file!")
        return None
    
    try: 
        df = pd.read_csv(dataset_path)
        print("Đã tải dữ liệu thành công!")
        return df
    except Exception as e:
        print(f"Đã xảy ra lỗi trong quá trình tải: {e}")
        return None
    
#define dataset path
dataset_path = "../data/dataset.csv"

#load data
df = load_dataset(dataset_path=dataset_path)

In [5]:
#Hiển thị tổng quan về các thông tin của dataset: số hàng, số cột, kiểu dữ liệu, số giá trị non-null mỗi cột, ...
print("Tổng quan cấu trúc dataset:")

#Thông tin số dòng số cột
print(f"- Dataset có {df.shape[0]} dòng và {df.shape[1]} cột.")

In [6]:
#Thông tin kiểu dữ liệu từng cột, số giá trị non-null
print("- Các thông tin cơ bản:")
df.info()

In [7]:
#Thông tin về trung bình, độ lệch chuẩn, tứ phân vị, giá trị nhỏ nhất lớn nhất
print("- Các thông tin thống kê numerical:")
display(df.describe())

print("- Các thông tin thống kê categorical:")
display(df.describe(include=['object', 'string']))

In [8]:
# Các cột numerical 
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
# Các cột categorical
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

In [9]:
#5 hàng dữ liệu đầu tiên
df.head()

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  

In [10]:
#Dùng hàm astype để convert tất cả dữ liệu trong cột từ bool thành int
df['target'] = (df['num'] > 0).astype(int)

#Hiển thị dữ liệu đã thêm target
df.head()

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  target  
0       fixed defect    0       0  
1             normal    2       1  
2  reversable defect    1       1  
3             normal    0       

In [11]:
def plot_target_distribution(data, column='target'):
    """
    Hàm vẽ biểu đồ đếm số lượng cho biến mục tiêu.
    data: DataFrame chứa dữ liệu
    column: Tên cột muốn vẽ (mặc định là 'target')
    """
    # Vẽ biểu đồ
    ax = sns.countplot(data=data, x=column, hue=column, palette="Set2", legend=True)

    # Gắn nhãn số lượng trên đầu mỗi cột
    for container in ax.containers:
        ax.bar_label(container, fontsize=10, fontweight='bold', padding=1)

    # Thiết lập tiêu đề và các trục
    counts = data[column].value_counts()
    ax.set_ylim(0, counts.max() * 1.2)
    plt.title('BIỂU ĐỒ THỐNG KÊ SỐ LƯỢNG NGƯỜI BỆNH VÀ KHÔNG BỆNH',
              fontsize=15,
              fontweight='bold',
              pad=20)
    plt.xlabel('Tình trạng sức khỏe')
    plt.ylabel('Số lượng người')

    # Hiển thị
    plt.show()


plot_target_distribution(df)

In [12]:
def plot_categorical(df, configs):
    """
    Vẽ các biểu đồ countplot cho danh sách các biến phân loại (Categorical).

    Hàm này tự động tạo ra một lưới các biểu đồ xếp dọc, gắn nhãn số lượng trên đầu mỗi cột
    và tinh chỉnh các thông số hiển thị để báo cáo trông chuyên nghiệp hơn.

    Args:
        df (pd.DataFrame): Bảng dữ liệu chứa các biến cần phân tích.
        configs (list of tuples): Danh sách cấu hình biểu đồ. Mỗi tuple bao gồm:
            (tên_cột_x, tên_cột_hue, tên_palette).
            Nếu x == hue, hàm sẽ hiểu là vẽ phân phối đơn biến.

    Returns:
        None: Hàm hiển thị biểu đồ trực tiếp bằng plt.show().
    """
    # Tạo khung hình động dựa trên số lượng biểu đồ
    fig, axes = plt.subplots(nrows=len(configs), ncols=1, figsize=(10, 6 * len(configs)))

    # Trường hợp chỉ có 1 biểu đồ, Matplotlib trả về 1 object thay vì list, cần ép về list
    if len(configs) == 1:
        axes = [axes]

    for i, (x_col, h_col, pal) in enumerate(configs):
        ax = axes[i]
        is_univariate = (x_col == h_col)

        # Vẽ biểu đồ chính
        sns.countplot(data=df, x=x_col, hue=h_col, palette=pal, ax=ax, legend=not is_univariate)

        # Gắn nhãn số lượng trên đầu mỗi cột
        for container in ax.containers:
            ax.bar_label(container, fontsize=11, fontweight='bold', padding=3)

        # Tinh chỉnh tiêu đề (Tự động nhận diện đơn biến hay đa biến)
        title_text = f'PHÂN PHỐI BIẾN {x_col.upper()}' if is_univariate else f'TỶ LỆ BỆNH THEO {x_col.upper()}'
        ax.set_title(title_text, pad=15, fontsize=14, fontweight='bold')
        ax.set_xlabel(x_col.capitalize(), fontsize=12)
        ax.set_ylabel('Count', fontsize=12)

        # Giữ chữ trên trục X nằm ngang cho dễ đọc
        if x_col in ['cp', 'restecg', 'slope', 'thal']:
            ax.tick_params(axis='x', rotation=0)

        # Chỉnh ylim để các nhãn bar_label không bị chạm đỉnh khung hình
        counts = df[x_col].value_counts()
        ax.set_ylim(0, counts.max() * 1.2)

    plt.tight_layout(pad=3.0)
    plt.show()

# --- Thực thi ---
my_configs = [
    ('sex', 'sex', 'Set2'),
    ('fbs','fbs','Set1' ),
    ('exang','exang', 'Set2'),
    ('cp', 'cp', 'plasma'),
    ('restecg','restecg','Set1' ),
    ('slope', 'slope', 'Set2'),
    ('thal','thal', 'plasma')
]

plot_categorical(df=df, configs=my_configs)

In [13]:
#hàm vẽ biểu đồ KDE cho nhiều cột trong bảng
def plot_distribution(df, num_cols, target='target'):
    """
    Vẽ biểu đồ KDE cho các cột numerical chia thành 2 đối tượng có bệnh và không bệnh bằng biến target
    Args:
        df (pandas.DataFrame): tập dữ liệu đầu vào
        cols (list): danh sách tên các cột cần vẽ biểu đồ
        target (str): tên cột target
    """
    #khung hình gồm 1 hàng và n cột mỗi cột là 1 biểu đồ, với n là số biểu đồ muốn vẽ
    fig, axes = plt.subplots(len(num_cols), 1, figsize=(20, 20))
    
    #lặp qua từng cột, i = vị trí, col = giá trị tên cột
    for i, col in enumerate(num_cols):
        #hàm vẽ biểu đồ KDE của thư viên seaborn
        sns.kdeplot(
            data=df,
            x=col,
            hue=target,
            ax=axes[i],
            fill=True,
            alpha=0.2,
            palette={0: "green", 1: "red"}
        )
        
        #Tiêu đề cho biểu đồ đang vẽ
        axes[i].set_title(col)
        
        #Nhãn của trục x
        axes[i].set_xlabel(col)
        
    plt.suptitle('Distribution of Numerical Features by Target',
                 fontsize=14, y=1.01)
    plt.tight_layout(pad=3.0)    
    
    # #Lưu hình ảnh biểu đồ
    # try:
    #     plt.savefig("../report/figures/distribution_analysis.png")
    #     print('Lưu hình ảnh thành công')
    # except Exception as e:
    #     print(f'Lỗi lưu ảnh: {e}')
        
    plt.show()

plot_distribution(df=df, num_cols=num_cols)

In [14]:
def encode_categorical(df, cat_cols):
    """
    Encode các categorical feature
    Args:
        df (pandas.DataFrame): tập dữ liệu đầu vào
        cat_cols (list): danh sách các cột là categorical
    Returns:
        (pandas.DataFrame): tập dữ liệu dã được encode
    """
    #tạo bản sao của df để encode mà không làm thay đổi dữ liệu gốc
    df_encoded = df.copy()
    
    #dùng Categorical().codes để encode thành số tự động
    for col in cat_cols:
        df_encoded[col] = pd.Categorical(df_encoded[col]).codes
    
    return df_encoded

def plot_correlation(df, target='target'):
    """
    Vẽ biểu đồ heatmap.
    Args:
        df (pandas.DataFrame): tập dữ liệu đầu vào
        target (str): tên cột target
    """

    #encode các cột categorical để tính hệ số pearson
    df_encoded = encode_categorical(df, cat_cols=cat_cols)
    
    #trong biểu đồ heatmap này các cột id, dataset không có ý nghĩa so sánh và cột num đã được chuyển thành target, nên không cần dùng các cột này
    df_encoded = df_encoded.drop(columns=['id', 'dataset', 'num'])
    
    #tính ma trận tương quan
    corr_matrix = df_encoded.corr()
    
    #vẽ heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        corr_matrix,
        annot=True,          
        fmt='.2f',           
        cmap='RdYlBu_r',     
        center=0,            
        vmin=-1, vmax=1,     
        linewidths=0.5,      
        square=True          
    )
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
    
    print('tương quan các cột với target theo độ mạnh:')
    print(corr_matrix[target].drop(target).sort_values(key=abs,ascending=False).round(2))
    
plot_correlation(df)

In [15]:
# Hàm vẽ Boxplot :  Boxplot giúp trực quan hóa phân bố dữ liệu và dễ dàng nhận ra các điểm nằm ngoài phạm vi bình thường.
def plot_boxplot(df, column):

    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[column])

    plt.title(f"Boxplot of {column}")
    plt.xlabel(column)

    plt.show()

In [16]:
# Phát hiện Outlier bằng phương pháp IQR : Phương pháp **Interquartile Range (IQR)** được sử dụng để xác định các giá trị bất thường.
def detect_outlier(df, column):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(f"\n===== Outlier in {column} =====")
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Number of outliers:", len(outliers))

    return outliers

In [17]:
# Phân tích Outlier cho từng biến:  Vẽ boxplot và Tính số lượng outlier bằng IQR
for col in num_cols:
    plot_boxplot(df, col)
    detect_outlier(df, col)

In [18]:
# Kiểm tra giá trị cực trị (Min và Max): Sau khi phát hiện outlier, chúng ta cần kiểm tra **giá trị nhỏ nhất và lớn nhất** để xác định: Outlier là dữ liệu thật hay lỗi nhập liệu
for col in num_cols:
    print(f"\nColumn: {col}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())

In [19]:
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)
df

      id  age     sex        dataset               cp  trestbps   chol    fbs  \
0      1   63    Male      Cleveland   typical angina     145.0  233.0   True   
1      2   67    Male      Cleveland     asymptomatic     160.0  286.0  False   
2      3   67    Male      Cleveland     asymptomatic     120.0  229.0  False   
3      4   37    Male      Cleveland      non-anginal     130.0  250.0  False   
4      5   41  Female      Cleveland  atypical angina     130.0  204.0  False   
..   ...  ...     ...            ...              ...       ...    ...    ...   
915  916   54  Female  VA Long Beach     asymptomatic     127.0  333.0   True   
916  917   62    Male  VA Long Beach   typical angina       NaN  139.0  False   
917  918   55    Male  VA Long Beach     asymptomatic     122.0  223.0   True   
918  919   58    Male  VA Long Beach     asymptomatic       NaN  385.0   True   
919  920   62    Male  VA Long Beach  atypical angina     120.0  254.0  False   

              restecg  thal

In [20]:
#Kết quả sau khi xử lý giá trị 0 thành NaN
for col in num_cols:
    detect_outlier(df, col)

In [21]:
df['target'].value_counts()

target
1    509
0    411
Name: count, dtype: int64

In [22]:
group_disease = df[df["target"] == 1]["age"]
group_no_disease = df[df["target"] == 0]["age"]

print("Số bệnh nhân có bệnh tim:", len(group_disease))
print("Số bệnh nhân không bệnh tim:", len(group_no_disease))

In [23]:
stat, p_value = stats.mannwhitneyu(group_disease, group_no_disease)

print("Statistic:", stat)
print("P-value:", p_value)

In [24]:
alpha = 0.05

print("\n===== Kết luận =====")

if p_value < alpha:
    print("Bác bỏ H0")
    print("→ Tuổi có ảnh hưởng đến nguy cơ bệnh tim")
else:
    print("Không đủ bằng chứng để bác bỏ H0")

In [25]:
missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
}).sort_values(by='missing_percent', ascending=False)

missing_summary

            column  missing_count  missing_percent
ca              ca            611        66.413043
thal          thal            486        52.826087
slope        slope            309        33.586957
chol          chol            202        21.956522
fbs            fbs             90         9.782609
oldpeak    oldpeak             62         6.739130
trestbps  trestbps             60         6.521739
thalch      thalch             55         5.978261
exang        exang             55         5.978261
restecg    restecg              2         0.217391
num            num              0         0.000000
id              id              0         0.000000
age            age              0         0.000000
cp              cp              0         0.000000
dataset    dataset              0         0.000000
sex            sex              0         0.000000
target      target              0         0.000000

In [26]:
plt.figure(figsize=(10,5))
sns.barplot(x='missing_percent', y='column', data=missing_summary)
plt.title('Tỷ lệ missing theo từng cột (%)')
plt.xlabel('Percent')
plt.ylabel('Column')
plt.show()

In [27]:
plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False)
plt.title('Heatmap Missing Values')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [28]:
# tổng số missing toàn dataset
total_missing = df.isnull().sum().sum()
print("Tổng số giá trị thiếu:", total_missing)

# cột nào > 50% missing
high_missing_cols = missing_summary[missing_summary['missing_percent'] > 50]
print("Cột có >50% missing:\n", high_missing_cols)

In [29]:
def feature_engineering(df):
    """
    Thực hiện tạo các đặc trưng mới dựa trên kiến thức y sinh.
    Args:
        df (pandas.DataFrame): tập dữ liệu đầu vào
    Returns:    
        pandas.DataFrame: tập dữ liệu sau khi đã được tạo đặc trưng mới
    """
    # Bước 1: Luôn copy dữ liệu gốc
    df_fe = df.copy()

    # Hàm nhỏ 1: Tính toán heart_rate_ratio
    def create_heart_rate_ratio(data):
        # Công thức Karvonen: thalch / (220 - age)
        return data['thalch'] / (220 - data['age'])

    # Hàm nhỏ 2: Phân nhóm tuổi nguy cơ
    def create_age_risk_group(data):
        bins = [0, 40, 50, 60, np.inf]
        labels = [0, 1, 2, 3]
        # Ép kiểu về int để Pipeline không lỗi
        return pd.cut(data['age'], bins=bins, labels=labels).astype(int)

    # Bước 2: Gọi các hàm nhỏ để gán giá trị vào DataFrame
    df_fe['heart_rate_ratio'] = create_heart_rate_ratio(df_fe)
    df_fe['age_risk_group'] = create_age_risk_group(df_fe)

    return df_fe

In [30]:
def plot_boxplot_heart_rate(df, configs):
    """
    Vẽ biểu đồ Boxplot kết hợp Stripplot để so sánh biến số theo Target.

    Args:
        df (pd.DataFrame): Bảng dữ liệu đã có feature mới.
        configs (list of tuples): Danh sách (tên_cột_số, tên_palette).
    """
    # Tạo khung hình dựa trên số lượng cấu hình truyền vào
    fig, axes = plt.subplots(nrows=len(configs), ncols=1, figsize=(10, 6 * len(configs)))

    # Ép về list nếu chỉ có 1 biểu đồ để vòng lặp for không bị lỗi
    if len(configs) == 1:
        axes = [axes]

    for i, (col, pal) in enumerate(configs):
        ax = axes[i]

        # 1. Vẽ Boxplot làm nền (thể hiện tứ phân vị và trung vị)
        sns.boxplot(data=df, x='target', y=col, palette=pal, hue='target', ax=ax, width=0.5)

        # 2. Vẽ thêm Stripplot (hiển thị từng chấm dữ liệu thực tế)
        sns.stripplot(data=df, x='target', y=col, color=".3", size=3, alpha=0.5, ax=ax)

        # 3. Tinh chỉnh
        ax.set_title(f'TỶ LỆ THEO NHÓM NGUY CƠ NHỊP TIM ({col.upper()}) ', pad=15, fontsize=14, fontweight='bold')
        ax.set_xlabel('Chẩn đoán (0: Khỏe, 1: Bệnh)', fontsize=12)
        ax.set_ylabel(col.replace('_', ' ').capitalize(), fontsize=12)
        ax.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout(pad=3.0)
    plt.show()


def plot_stacked_age_risk(df, col):
    """
    Vẽ biểu đồ Stacked Bar Chart cho age_risk_group .
    """
    # 1. Tính toán tỷ lệ % (Màu xanh = khỏe, Màu cam/đỏ = bệnh)
    data_pct = pd.crosstab(df[col], df['target'], normalize='index') * 100

    # 2. Vẽ biểu đồ cột chồng
    ax = data_pct.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#66c2a5', '#fc8d62'])

    # 3. Tinh chỉnh
    plt.title(f'TỶ LỆ BỆNH THEO NHÓM NGUY CƠ TUỔI ({col.upper()})', pad=15, fontsize=14, fontweight='bold')
    plt.xlabel('Nhóm tuổi (0:[0–40],1:[40–50] ,2:[50–60],3:[60+])', fontsize=12)
    plt.ylabel('Phần trăm (%)', fontsize=12)
    plt.xticks(rotation=0)
    plt.legend(['Khỏe (Target 0)', 'Bệnh (Target 1)'], loc='upper right', title='Trạng thái')

    # Gắn nhãn số % lên cột
    for p in ax.patches:
        width, height = p.get_width(), p.get_height()
        x, y = p.get_xy()
        if height > 0:
            ax.text(x + width / 2, y + height / 2, f'{height:.1f}%', ha='center', va='center', color='white',
                    fontweight='bold')

    plt.tight_layout()
    plt.show()


####################
df_fe = feature_engineering(df)

# BIỂU ĐỒ 1: Boxplot cho biến số (heart_rate_ratio)
print("---VẼ BIỂU ĐỒ 1 (BOXPLOT) ---")
my_heart_rate_configs = [('heart_rate_ratio', 'Set2')]
plot_boxplot_heart_rate(df=df_fe, configs=my_heart_rate_configs)

# BIỂU ĐỒ 2: Stacked Bar cho biến phân loại (age_risk_group)
print("--- VẼ BIỂU ĐỒ 2 (STACKED BAR) ---")
plot_stacked_age_risk(df=df_fe, col='age_risk_group')

In [31]:
def evaluate_model_improvement(X_train, y_train):
    """
    So sánh F1 trước/sau FE bằng 5-fold CV.
    Dùng ColumnTransformer nhất quán với Pipeline chính thức.
    """
    from sklearn.model_selection import StratifiedKFold

    # Cột gốc (chưa FE)
    thal_col = 'thalch' if 'thalch' in X_train.columns else 'thalach'
    original_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                     'restecg', thal_col, 'exang', 'oldpeak', 
                     'slope', 'ca', 'thal']
    existing_cols = [c for c in original_cols if c in X_train.columns]

    # Phân nhóm cột gốc
    num_base = [c for c in existing_cols 
                if c in ['age','trestbps','chol',thal_col,'oldpeak','ca']]
    cat_base = [c for c in existing_cols 
                if c in ['sex','cp','fbs','restecg','exang','slope','thal']]

    # Phân nhóm cột có FE
    num_fe = num_base + ['heart_rate_ratio', 'age_risk_group']
    cat_fe = cat_base

    def build_pipeline(num_cols, cat_cols):
        # Dùng đúng ColumnTransformer như Section XI
        pre = ColumnTransformer([
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)
        ])
        return Pipeline([
            ('preprocessor', pre),
            ('lr', LogisticRegression(max_iter=1000, 
                                      random_state=RANDOM_STATE))
        ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, 
                         random_state=RANDOM_STATE)

    f1_base = cross_val_score(
        build_pipeline(num_base, cat_base),
        X_train[existing_cols], y_train,
        cv=cv, scoring='f1'
    ).mean()

    f1_new = cross_val_score(
        build_pipeline(num_fe, cat_fe),
        X_train, y_train,
        cv=cv, scoring='f1'
    ).mean()

    print(f"\n--- KẾT QUẢ ĐÁNH GIÁ ĐỊNH LƯỢNG ---")
    print(f"F1 Baseline (Gốc): {f1_base:.7f}")
    print(f"F1 After FE (Mới): {f1_new:.7f}")
    print(f"Cải thiện: {(f1_new - f1_base) * 100:.5f}%")

In [32]:
# Tách X và y
drop_cols = ['id', 'num', 'dataset', 'target']

X = df.drop(columns=drop_cols)
y = df['target']

# In thông tin cơ bản về X và y để kiểm tra trước khi FE
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nCác cột trong X:")
print(X.columns.tolist())
print("\nCác giá trị duy nhất của y:")
print(y.unique())

In [33]:
# Áp dụng feature engineering
X_fe = feature_engineering(X)

# In thông tin sau khi FE để kiểm tra
print("X shape trước FE:", X.shape)
print("X shape sau FE:", X_fe.shape)
print("Các cột sau FE:", X_fe.columns.tolist())
print("Các features mới sau FE:")
print(X_fe.head()[['heart_rate_ratio', 'age_risk_group']])

In [34]:
# Tách tập Train/Test với stratify để giữ tỷ lệ class giữa 2 tập
X_train, X_test, y_train, y_test = train_test_split(X_fe, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# In thông tin về kích thước tập Train/Test để kiểm tra
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [35]:
# In thông tin về tỷ lệ phần trăm kích thước tập Train/Test và tỷ lệ class trong mỗi tập để kiểm tra stratify có hoạt động đúng không
print("Kích thước tập Train:", y_train.shape[0]/y.shape[0] * 100, "%")
print("Kích thước tập Test:", y_test.shape[0]/y.shape[0] * 100, "%")
print()

# Kiểm tra tỷ lệ class trong y_train và y_test để đảm bảo stratify hoạt động đúng
print("Tỷ lệ class trong y_train:")
print(y_train.value_counts(normalize=True).round(3))
print()

print("Tỷ lệ class trong y_test:")
print(y_test.value_counts(normalize=True).round(3))

In [36]:
#lấy danh sách cột numerical và categorical sau khi đã có FE để tiện sử dụng cho các bước tiếp theo
num_cols_fe = num_cols + ['heart_rate_ratio', 'age_risk_group']
print(num_cols_fe)

cat_cols_fe = cat_cols
print(cat_cols_fe)

print("Numerical  :", len(num_cols_fe), "cột")
print("Categorical:", len(cat_cols_fe), "cột")
print("Tổng       :", len(num_cols_fe) + len(cat_cols_fe), "cột")

In [37]:
#Pipe line cho cột numerical
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#Pipe line cho cột categorical
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [38]:
# Định nghĩa ColumnTransformer để áp dụng các pipeline tương ứng cho từng nhóm cột
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols_fe),
    ('cat', cat_pipeline, cat_cols_fe)
])

print(preprocessor)

In [39]:
# Áp dụng preprocessor cho tập Train để chuẩn bị dữ liệu cho mô hình
preprocessor.fit(X_train)

# Biến đổi cả tập Train và Test bằng preprocessor đã fit
X_train_prepared = preprocessor.transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

In [40]:
# Shape trước và sau
print("Shape X_train trước:", X_train.shape)
print("Shape X_train sau:  ", X_train_prepared.shape)
print("Shape X_test trước: ", X_test.shape)
print("Shape X_test sau:   ", X_test_prepared.shape)

print()

# Kiểm tra không còn NaN
print("NaN X_train_prepared:", np.isnan(X_train_prepared).sum())
print("NaN X_test_prepared: ", np.isnan(X_test_prepared).sum())

print()

# Kiểm tra dtype
print("Dtype:", X_train_prepared.dtype)

In [41]:
print("\n--- Dữ liệu đã được chuẩn bị sẵn sàng cho mô hình ---")
print("X_train_prepared shape:", X_train_prepared.shape)
print("y_train shape:", y_train.shape)
print("X_train_prepared:\n", X_train_prepared)

In [42]:
evaluate_model_improvement(X_train, y_train)

In [43]:
#Hàm log_model_to_wandb để lưu mô hình và kết quả đánh giá vào WandB
def log_model_to_wandb(model, model_name, config, y_test, y_pred, y_proba, metrics):
    """
    Tạo run, lưu mô hình và kết quả đánh giá vào WandB.
    """

    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=model_name,
        config=config,
        reinit=True,
        settings=wandb.Settings(init_timeout=180)
    ) as run:

        # Log metrics
        wandb.log(metrics)

        # Confusion matrix
        wandb.log({
            "confusion_matrix": wandb.plot.confusion_matrix(
                probs=None,
                y_true=list(y_test.reset_index(drop=True)),
                preds=list(y_pred),
                class_names=['No Disease', 'Disease']
            )
        })

        # ROC + PR curve
        if y_proba is not None:
            y_proba_2d = np.column_stack([1 - y_proba, y_proba])

            wandb.log({
                "roc_curve": wandb.plot.roc_curve(
                    y_true=list(y_test.reset_index(drop=True)),
                    y_probas=y_proba_2d,
                    labels=['No Disease', 'Disease']
                ),
                "pr_curve": wandb.plot.pr_curve(
                    y_true=list(y_test.reset_index(drop=True)),
                    y_probas=y_proba_2d,
                    labels=['No Disease', 'Disease']
                )
            })

        # Save model
        model_filename = f"{model_name.replace(' ', '_').lower()}_model.pkl"
        jb.dump(model, model_filename)

        model_artifact = wandb.Artifact(
            model_name.replace(' ', '_').lower() + "_model",
            type='model'
        )
        model_artifact.add_file(model_filename)
        run.log_artifact(model_artifact)

        os.remove(model_filename)

        print(f"Đã log mô hình '{model_name}' và kết quả đánh giá vào WandB thành công!")

In [44]:
# Hàm evaluate_model để đánh giá mô hình, vẽ biểu đồ và trả về số liệu
def evaluate_model(model, model_name, X_test, y_test, threshold=0.5):

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        y_prob_positive = y_prob[:, 1]
    else:
        y_prob = None
        y_prob_positive = model.predict(X_test)

    y_pred = (y_prob_positive >= threshold).astype(int)

    test_recall = recall_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_prob_positive)

    print(f"Kết quả Test (Threshold = {threshold}):")
    print(f"   - Recall:    {test_recall:.4f}")
    print(f"   - Precision: {test_precision:.4f}")
    print(f"   - F1-Score:  {test_f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Predict 0', 'Predict 1'],
        yticklabels=['Actual 0', 'Actual 1']
    )

    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Thực tế (Actual)')
    plt.xlabel('Dự đoán (Predicted)')
    plt.tight_layout()
    plt.show()

    metrics_dict = {
        "Test_Recall": test_recall,
        "Test_Precision": test_precision,
        "Test_F1": test_f1,
        "Test_ROC_AUC": test_auc,
        "Threshold_used": threshold
    }

    return metrics_dict, y_pred, y_prob_positive

In [45]:
# Hàm tổng để Train -> Evaluate -> Log lên WandB
def train_and_run_experiment(
    model,
    model_name,
    config,
    X_train,
    y_train,
    X_test,
    y_test,
    threshold=0.5,
    random_state=RANDOM_STATE
):

    print(f"Training {model_name} ...")

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='recall',
        n_jobs=-1
    )

    cv_recall_mean = cv_scores.mean()
    print(f"CV Recall (5-Fold): {cv_recall_mean:.4f}")

    model.fit(X_train, y_train)

    metrics_dict, y_pred, y_prob = evaluate_model(
        model=model,
        model_name=model_name,
        X_test=X_test,
        y_test=y_test,
        threshold=threshold
    )

    print(f"Metrics trên tập Test đã được tính toán: {metrics_dict}")

    metrics_dict["CV_Recall_Mean"] = cv_recall_mean

    log_model_to_wandb(
        model=model,
        model_name=model_name,
        config=config,
        y_test=y_test,
        y_pred=y_pred,
        y_proba=y_prob,
        metrics=metrics_dict
    )

    print(f"Đã log mô hình và kết quả đánh giá của {model_name} lên WandB thành công!")
    print(f"Đã hoàn thành experiment cho {model_name}.\n")

    return model, metrics_dict

In [46]:
#khởi tạo model Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')

In [47]:
#khai báo pram_grid cho GridSearchCV
param_grid_lr = [
    {
        'C': [0.01, 0.1, 1, 10], 
        'penalty': ['l1', 'l2'], 
        'solver': ['liblinear']   # liblinear hỗ trợ cả l1 và l2
    },
    {
        'C': [0.01, 0.1, 1, 10], 
        'penalty': ['l2'],        # lbfgs chỉ hỗ trợ l2
        'solver': ['lbfgs']
    }
]

In [48]:
#khởi tạo GridSearchCV cho Logistic Regression với 5-fold CV, scoring là recall, 
#sử dụng tất cả CPU có sẵn và verbose để theo dõi tiến trình
grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

In [49]:
#fit GridSearchCV cho Logistic Regression
grid_lr.fit(X_train_prepared, y_train)  

    
print("Best params:", grid_lr.best_params_)
print("Best CV Recall:", grid_lr.best_score_)

In [50]:
#Chạy experiment với model Logistic Regression đã được tối ưu hyperparameters từ GridSearchCV, 
#sử dụng threshold 0.4 để đánh giá trên tập Test và log kết quả lên WandB với tên "Logistic Regression"                                                                                                                                                               
lr_model, lr_metrics = train_and_run_experiment(
    model=grid_lr.best_estimator_,
    model_name="Logistic Regression",
    config=grid_lr.best_params_,
    X_train=X_train_prepared,
    y_train=y_train,
    X_test=X_test_prepared,
    y_test=y_test,
    threshold=0.4
)

In [51]:
#khởi tạo Random forest
rf_config = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE
}

rf_model = RandomForestClassifier(**rf_config)

In [52]:
#Khai báo param_grid cho Random Forest
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

In [53]:
#GridSearchCV Random Forest
rf = RandomForestClassifier(random_state=RANDOM_STATE)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_prepared, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV Recall:", grid_rf.best_score_)

In [54]:
#Run 1 (GridSearchCV)
best_params_rf = grid_rf.best_params_

rf_model, rf_metrics = train_and_run_experiment(
    model=RandomForestClassifier(**best_params_rf, random_state=RANDOM_STATE),
    model_name="Random Forest",
    config=best_params_rf,
    X_train=X_train_prepared,
    y_train=y_train,
    X_test=X_test_prepared,
    y_test=y_test,
    threshold=0.4
)